# Leer lugares y personajes

In [58]:
import re
import unicodedata
import pandas as pd

In [59]:
ruta_lugares = "../data/external/terminos/Lugares literarios.xlsx"
df_lugares = pd.read_excel(ruta_lugares)
df_lugares.head()

,Autor,Obra principal,Lugar ficticio,Descripción breve del lugar,Lugar_nombres
0,J.K. Rowling,Harry Potter,Callejón Diagon (Diagon Alley),La calle mágica oculta en Londres donde los ma...,"Callejón Diagon, Diagon Alley, Diagon"
1,A. A. Milne,Winnie the Pooh,Bosque de los Cien Acres (Hundred Acre Wood),"El hogar de Pooh, Piglet, Tigger y sus amigos,...","Bosque de los Cien Acres, Hundred Acre Wood"
2,Aldous Huxley,Un mundo feliz,El Estado Mundial (World State),"La sociedad distópica global de la novela, bas...",Estado Mundial
3,C. S. Lewis,Las Crónicas de Narnia,Narnia,Un reino mágico al que se accede a través de u...,Narnia
4,Carlo Collodi,Las aventuras de Pinocho,País de los juguetes (o Juguetelandia),La isla de la tentación donde los niños que no...,"País de los juguetes, Juguetelandia"


In [60]:
ruta_personajes = "../data/external/terminos/Personajes literarios.xlsx"
df_personajes = pd.read_excel(ruta_personajes)
df_personajes.head()

,Autor,Obra principal,Personaje ficticio,Rol principal / característica,Personajes_nombres
0,A. A. Milne,Winnie the Pooh,Winnie the Pooh,"El oso tierno y despistado, amante de la miel.","Winnie the Pooh, Pooh, Winnie Pooh"
1,Agatha Christie,Novelas de Poirot,Hércules Poirot,"El detective belga, famoso por sus ""pequeñas c...","Hércules Poirot, Poirot"
2,Agatha Christie,Novelas de Miss Marple,Miss Marple,La anciana observadora que resuelve crímenes e...,"Miss Marple, Marple"
3,Albert Camus,El Extranjero,Meursault,"El protagonista indiferente, símbolo del absur...",Meursault
4,Albert Camus,La peste,Dr. Bernard Rieux,"El médico que lucha contra la epidemia, símbol...","Dr. Bernard Rieux, Bernard Rieux, Rieux"


In [61]:
len(df_lugares), len(df_personajes)

(53, 118)

In [62]:
# Combinar obras de df_personajes y df_lugares
df_obras = pd.concat([
    df_personajes[['Autor', 'Obra principal']],
    df_lugares[['Autor', 'Obra principal']]
], ignore_index=True)

# Eliminar duplicados
df_obras = df_obras.drop_duplicates().reset_index(drop=True)

print(f"Obras únicas encontradas: {len(df_obras)}")
df_obras.head()

Obras únicas encontradas: 128


,Autor,Obra principal
0,A. A. Milne,Winnie the Pooh
1,Agatha Christie,Novelas de Poirot
2,Agatha Christie,Novelas de Miss Marple
3,Albert Camus,El Extranjero
4,Albert Camus,La peste


In [63]:
df_chunks = pd.read_parquet("../data/processed/chunks.parquet")
df_chunks.head()

,id_doc,autor_doc,fecha_doc,diario_doc,titulo_doc,chunk_id,texto_chunk
0,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,0,"La Coalición Colombia Partido Alianza Verde, P..."
1,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,1,"al mismo tiempo lo exponen, en ciertas ocasion..."
2,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,2,los acuerdos con las Farc. Anunció que no prom...
3,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,3,moratoria en la explotación tipo fracking. Y f...
4,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,0,Las interpretaciones de la historia sirven com...


# Búsqueda exacta

In [64]:
# ---------------------------------------------------------------------------
# Normalización
# ---------------------------------------------------------------------------

def normalizar_texto(texto: str) -> str:
    """Convierte a minúsculas, elimina tildes y puntos."""
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = texto.replace('.', '')
    return texto



In [65]:
# Función de búsqueda ---------------------

from semver import match


def buscar_terminos(df_terminos: pd.DataFrame,
                    col_nombres: str,
                    col_canon: str,
                    col_salida: str,
                    df_chunks: pd.DataFrame) -> pd.DataFrame:
    """
    Busca términos literarios (lugares o personajes) en los chunks de texto.

    Parámetros
    ----------
    df_terminos : DataFrame con 'Obra principal', col_nombres y col_canon.
    col_nombres : Columna con variantes de nombre separadas por coma
                  (ej. 'Lugar_nombres' o 'Personajes_nombres').
    col_canon   : Columna con el nombre canónico del término
                  (ej. 'Lugar ficticio' o 'Personaje ficticio').
    col_salida  : Nombre de la columna resultado
                  (ej. 'Lugar_encontrado' o 'Personaje_encontrado').
    df_chunks   : DataFrame con 'id_doc', 'chunk_id', 'texto_chunk'.

    Retorna
    -------
    DataFrame con: col_salida, 'Obra principal', 'id_doc', 'chunk_id', 'texto_chunk'.
    """

    # 1. Normalizar todos los chunks (vectorizado con .map)
    chunks_norm = df_chunks['texto_chunk'].map(normalizar_texto).to_numpy()
    id_docs     = df_chunks['id_doc'].to_numpy()
    chunk_ids   = df_chunks['chunk_id'].to_numpy()
    textos_orig = df_chunks['texto_chunk'].to_numpy()

    resultados = []

    for _, fila in df_terminos.iterrows():
        nombre_canon = fila[col_canon]
        obra         = fila['Obra principal']

        # Variantes del nombre separadas por coma
        variantes_norm = [
            normalizar_texto(v.strip())
            for v in str(fila[col_nombres]).split(',')
            if v.strip()
        ]

        # Patrón que detecta CUALQUIER variante como palabra completa
        patron = re.compile(
            r'\b(' + '|'.join(re.escape(v) for v in variantes_norm) + r')\b'
        )

        # 2. Buscar en cada chunk y registrar coincidencias
        for idx in range(len(chunks_norm)):
            match = patron.search(chunks_norm[idx])
            if match:
                resultados.append({
                    col_salida:       nombre_canon,
                    'Variante_hallada': match.group(0),
                    'Obra principal': obra,
                    'id_doc':         id_docs[idx],
                    'chunk_id':       chunk_ids[idx],
                    'texto_chunk':    textos_orig[idx],
                })

    df_resultado = pd.DataFrame(resultados)

    # 3. Deduplicar: misma entidad en el mismo chunk (salvaguarda)
    if not df_resultado.empty:
        df_resultado = (
            df_resultado
            .drop_duplicates(subset=[col_salida, 'id_doc', 'chunk_id'])
            .reset_index(drop=True)
        )

    return df_resultado


In [66]:
# Buscar lugares

df_res_lugares = buscar_terminos(
    df_terminos = df_lugares,
    col_nombres = 'Lugar_nombres',
    col_canon   = 'Lugar ficticio',
    col_salida  = 'Lugar_encontrado',
    df_chunks   = df_chunks,
)

print(f"Lugares encontrados: {len(df_res_lugares):,} filas")
df_res_lugares.head()

Lugares encontrados: 252 filas


,Lugar_encontrado,Variante_hallada,Obra principal,id_doc,chunk_id,texto_chunk
0,Narnia,narnia,Las Crónicas de Narnia,11303,3,acontecimientos inverosímiles no perturban más...
1,La Mancha,la mancha,Don Quijote de la Mancha,991,0,Pese a que exhalaciones empezaron a registrars...
2,La Mancha,la mancha,Don Quijote de la Mancha,1445,2,y cada día lucharía contra ellos con un conoci...
3,La Mancha,la mancha,Don Quijote de la Mancha,1950,7,porvenir sino modificaciones fecundas en lo in...
4,La Mancha,la mancha,Don Quijote de la Mancha,2960,0,El primer Brexit en la historia de Inglaterra ...


In [67]:
# Buscar personajes

df_res_personajes = buscar_terminos(
    df_terminos = df_personajes,
    col_nombres = 'Personajes_nombres',
    col_canon   = 'Personaje ficticio',
    col_salida  = 'Personaje_encontrado',
    df_chunks   = df_chunks,
)

print(f"Personajes encontrados: {len(df_res_personajes):,} filas")
df_res_personajes.head()

Personajes encontrados: 995 filas


,Personaje_encontrado,Variante_hallada,Obra principal,id_doc,chunk_id,texto_chunk
0,Winnie the Pooh,winnie pooh,Winnie the Pooh,5916,0,¿Qué pasará cuando los censores chinos traten ...
1,Winnie the Pooh,winnie pooh,Winnie the Pooh,5916,3,nación en la historia. En lo que respecta a Ch...
2,Winnie the Pooh,winnie pooh,Winnie the Pooh,5916,7,"es el mayor temor de Xi, y no deberíamos despe..."
3,Hércules Poirot,poirot,Novelas de Poirot,3350,1,su involucramiento con carteles mexicanos. En ...
4,Hércules Poirot,hercules poirot,Novelas de Poirot,9425,0,"Después de Shakespeare, la autora más leída en..."


In [68]:
# Buscar autores
df_res_autor = buscar_terminos(
    df_terminos = df_obras, 
    col_nombres = 'Autor',
    col_canon   = 'Autor',
    col_salida  = 'Autor_encontrado',
    df_chunks   = df_chunks,
)
print(f"Autores encontrados: {len(df_res_autor):,} filas")
df_res_autor.head()

Autores encontrados: 756 filas


,Autor_encontrado,Variante_hallada,Obra principal,id_doc,chunk_id,texto_chunk
0,Agatha Christie,agatha christie,Novelas de Poirot,8008,0,"La amenaza de contagio por el coronavirus, con..."
1,Agatha Christie,agatha christie,Novelas de Poirot,8716,1,alguna de su vasta y promiscua trayectoria cri...
2,Agatha Christie,agatha christie,Novelas de Poirot,9425,0,"Después de Shakespeare, la autora más leída en..."
3,Agatha Christie,agatha christie,Novelas de Poirot,9425,2,comenzado a transitar por el camino de su inme...
4,Agatha Christie,agatha christie,Novelas de Poirot,11507,0,Pasan los días y el caso de Jesús Santrich sig...


In [69]:
# Buscar obras
df_res_obras = buscar_terminos(
    df_terminos = df_obras, 
    col_nombres = 'Obra principal',
    col_canon   = 'Obra principal',
    col_salida  = 'Obra_encontrada',
    df_chunks   = df_chunks,
)
print(f"Obras encontradas: {len(df_res_obras):,} filas")
df_res_obras.head()

Obras encontradas: 888 filas


,Obra_encontrada,Variante_hallada,Obra principal,id_doc,chunk_id,texto_chunk
0,El Extranjero,el extranjero,El Extranjero,213,1,que sirven de escenografía a enternecedoras ha...
1,El Extranjero,el extranjero,El Extranjero,332,4,a la repatriación de capitales. La reacción de...
2,El Extranjero,el extranjero,El Extranjero,401,1,Lo impredecible y errático de Trump se debe en...
3,El Extranjero,el extranjero,El Extranjero,403,1,me parece que el Eln no tiene la intención de ...
4,El Extranjero,el extranjero,El Extranjero,407,0,La reforma tributaria aprobada por el Congreso...


In [70]:
## Guardar resultados -----

# Asegurar que todas las columnas de texto sean string (evita ArrowTypeError)
for col in ['Lugar_encontrado', 'Obra principal', 'texto_chunk']:
    df_res_lugares[col] = df_res_lugares[col].astype(str)

for col in ['Personaje_encontrado', 'Obra principal', 'texto_chunk']:
    df_res_personajes[col] = df_res_personajes[col].astype(str)

for col in ['Obra_encontrada', 'Obra principal', 'texto_chunk']:
    df_res_obras[col] = df_res_obras[col].astype(str)

df_res_lugares.to_parquet("../data/processed/resultados_lugares.parquet",        index=False)
df_res_personajes.to_parquet("../data/processed/resultados_personajes.parquet",  index=False)
df_res_obras.to_parquet("../data/processed/resultados_obras.parquet",  index=False)

df_res_lugares.to_excel("../data/processed/resultados_lugares_literarios_exacta.xlsx",        index=False)
df_res_personajes.to_excel("../data/processed/resultados_personajes_literarios_exacta.xlsx",  index=False)
df_res_obras.to_excel("../data/processed/resultados_obras_literarios_exacta.xlsx",  index=False)
df_res_autor.to_excel("../data/processed/resultados_autores_literarios_exacta.xlsx",  index=False)

print("Archivos guardados correctamente.")

Archivos guardados correctamente.
